# Bài 4 - Gom cụm D1 và D2

Notebook này thực hiện:

- Chuẩn hóa/co giãn dữ liệu số và mã hóa thuộc tính hạng mục của D1.
- K-means (thuật toán 1), chọn số cụm bằng elbow và silhouette.
- DBSCAN (thuật toán 2), đánh giá bằng silhouette và Davies-Bouldin trên các điểm không bị xem là nhiễu.
- Profiling cụm và xuất bảng kết quả, hình minh họa và `bao-cao-bai4.pdf`.

D1 và D2 được phân tích riêng vì có đơn vị quan sát khác nhau: một dòng đặt phòng ở D1 và một khách hàng ở D2.

In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.decomposition import TruncatedSVD
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import silhouette_score, davies_bouldin_score

ROOT = Path.cwd().parent
D1_PATH = ROOT / "bai1-du-lieu-tien-xu-ly" / "D1" / "data" / "processed" / "hotel_bookings_cleaned.csv"
D2_PATH = ROOT / "bai1-du-lieu-tien-xu-ly" / "D2" / "data" / "processed" / "customer_features.csv"
OUT = Path.cwd() / "outputs"
FIG = OUT / "figures"
OUT.mkdir(exist_ok=True)
FIG.mkdir(exist_ok=True)

print("D1:", D1_PATH.exists(), D1_PATH)
print("D2:", D2_PATH.exists(), D2_PATH)
print("Output:", OUT.resolve())

In [ ]:
def make_encoder():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=True)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=True)


def evaluate_labels(X, labels):
    labels = np.asarray(labels)
    mask = labels != -1
    unique = np.unique(labels[mask])
    if len(unique) < 2 or mask.sum() < 3:
        return np.nan, np.nan, int(mask.sum()), int((labels == -1).sum())
    return silhouette_score(X[mask], labels[mask]), davies_bouldin_score(X[mask], labels[mask]), int(mask.sum()), int((labels == -1).sum())


def choose_k(X, ks=range(2, 9), random_state=42):
    rows = []
    for k in ks:
        model = KMeans(n_clusters=k, n_init=10, random_state=random_state)
        labels = model.fit_predict(X)
        rows.append({"k": k, "inertia": model.inertia_, "silhouette": silhouette_score(X, labels)})
    return pd.DataFrame(rows)


def save_selection_plot(scores, title, filename):
    fig, ax1 = plt.subplots(figsize=(8, 4.5))
    ax1.plot(scores["k"], scores["inertia"], "o-", color="#176b87")
    ax1.set_xlabel("Số cụm k")
    ax1.set_ylabel("Inertia", color="#176b87")
    ax2 = ax1.twinx()
    ax2.plot(scores["k"], scores["silhouette"], "s-", color="#d95f02")
    ax2.set_ylabel("Silhouette", color="#d95f02")
    ax1.set_title(title)
    fig.tight_layout()
    fig.savefig(FIG / filename, dpi=160)
    plt.show()
    plt.close(fig)


def profile_sample(frame, labels, name):
    prof = frame.copy().reset_index(drop=True)
    prof["cluster"] = labels
    numeric = prof.select_dtypes(exclude=["object"]).columns.drop("cluster", errors="ignore")
    numeric_profile = prof.groupby("cluster", dropna=False)[list(numeric)].mean().round(3)
    counts = prof["cluster"].value_counts().sort_index().rename("n").to_frame()
    categorical_rows = []
    for cluster, group in prof.groupby("cluster", dropna=False):
        row = {"cluster": cluster, "n": len(group)}
        for col in prof.select_dtypes(include=["object"]).columns:
            mode = group[col].mode(dropna=True)
            row[f"top_{col}"] = mode.iloc[0] if len(mode) else "NA"
        categorical_rows.append(row)
    categorical_profile = pd.DataFrame(categorical_rows).set_index("cluster")
    numeric_profile.to_csv(OUT / f"{name}_numeric_profile.csv")
    categorical_profile.to_csv(OUT / f"{name}_categorical_profile.csv")
    return counts.join(numeric_profile, how="left"), categorical_profile

## D1 - Hotel Booking Demand

Các thuộc tính hạng mục (`hotel`, `meal`, `country`, `market_segment`, `distribution_channel`, `reserved_room_type`, `assigned_room_type`, `deposit_type`, `customer_type`) được mã hóa one-hot. Thuộc tính số được chuẩn hóa bằng z-score. TruncatedSVD giảm biểu diễn thưa xuống 20 chiều để các thuật toán khoảng cách hoạt động ổn định hơn.

In [ ]:
d1 = pd.read_csv(D1_PATH)
d1 = d1.drop(columns=["is_canceled"], errors="ignore")
cat_d1 = d1.select_dtypes(include=["object"]).columns.tolist()
num_d1 = d1.select_dtypes(exclude=["object"]).columns.tolist()
pre_d1 = ColumnTransformer([("num", StandardScaler(), num_d1), ("cat", make_encoder(), cat_d1)])
X1_sparse = pre_d1.fit_transform(d1)
n_svd = min(20, X1_sparse.shape[1] - 1)
svd_d1 = TruncatedSVD(n_components=n_svd, random_state=42)
X1 = svd_d1.fit_transform(X1_sparse)
rng = np.random.RandomState(42)
sample_size = min(5000, len(d1))
sample_idx = rng.choice(len(d1), size=sample_size, replace=False)
X1_sample = X1[sample_idx]
print(f"D1: {d1.shape[0]:,} dòng; {len(num_d1)} số; {len(cat_d1)} hạng mục; clustering: {X1.shape[1]} chiều")
print(f"Mẫu đánh giá: {sample_size:,} dòng")

In [ ]:
d1_scores = choose_k(X1_sample)
d1_scores.to_csv(OUT / "d1_k_selection.csv", index=False)
save_selection_plot(d1_scores, "D1 - Chọn k cho K-means", "d1_k_selection.png")
best_k_d1 = int(d1_scores.loc[d1_scores["silhouette"].idxmax(), "k"])
km1 = KMeans(n_clusters=best_k_d1, n_init=10, random_state=42)
d1_kmeans_labels = km1.fit_predict(X1_sample)
db1 = DBSCAN(eps=1.2, min_samples=12, n_jobs=-1)
d1_dbscan_labels = db1.fit_predict(X1_sample)
s_km, db_km, _, _ = evaluate_labels(X1_sample, d1_kmeans_labels)
s_db, db_db, kept_db, noise_db = evaluate_labels(X1_sample, d1_dbscan_labels)
d1_metrics = pd.DataFrame([
    {"dataset": "D1", "algorithm": "K-means", "clusters": best_k_d1, "silhouette": s_km, "davies_bouldin": db_km, "evaluated_points": len(d1_kmeans_labels), "noise_points": 0},
    {"dataset": "D1", "algorithm": "DBSCAN", "clusters": len(set(d1_dbscan_labels)) - (1 if -1 in d1_dbscan_labels else 0), "silhouette": s_db, "davies_bouldin": db_db, "evaluated_points": kept_db, "noise_points": noise_db},
])
d1_metrics.to_csv(OUT / "d1_clustering_metrics.csv", index=False)
d1_metrics

In [ ]:
d1_k_profile, d1_k_cat = profile_sample(d1.iloc[sample_idx].reset_index(drop=True), d1_kmeans_labels, "d1_kmeans")
d1_db_profile, d1_db_cat = profile_sample(d1.iloc[sample_idx].reset_index(drop=True), d1_dbscan_labels, "d1_dbscan")
print("D1 K-means profile")
display(d1_k_profile)
print("D1 DBSCAN categorical profile")
display(d1_db_cat)

## D2 - Instacart customer features

D2 đã được chuyển từ giao dịch sang một dòng/khách hàng trong Bài 1. Các thuộc tính đều là số nên dùng StandardScaler trực tiếp, không one-hot.

In [ ]:
d2 = pd.read_csv(D2_PATH)
feature_d2 = [c for c in d2.columns if c != "user_id"]
d2 = d2.dropna(subset=feature_d2).copy()
scaler_d2 = StandardScaler()
X2 = scaler_d2.fit_transform(d2[feature_d2])
rng2 = np.random.RandomState(42)
sample_size2 = min(5000, len(d2))
sample_idx2 = rng2.choice(len(d2), size=sample_size2, replace=False)
X2_sample = X2[sample_idx2]
print(f"D2: {d2.shape[0]:,} khách hàng; {len(feature_d2)} đặc trưng số")

In [ ]:
d2_scores = choose_k(X2_sample)
d2_scores.to_csv(OUT / "d2_k_selection.csv", index=False)
save_selection_plot(d2_scores, "D2 - Chọn k cho K-means", "d2_k_selection.png")
best_k_d2 = int(d2_scores.loc[d2_scores["silhouette"].idxmax(), "k"])
km2 = KMeans(n_clusters=best_k_d2, n_init=10, random_state=42)
d2_kmeans_labels = km2.fit_predict(X2_sample)
db2 = DBSCAN(eps=0.85, min_samples=12, n_jobs=-1)
d2_dbscan_labels = db2.fit_predict(X2_sample)
s_km, db_km, _, _ = evaluate_labels(X2_sample, d2_kmeans_labels)
s_db, db_db, kept_db, noise_db = evaluate_labels(X2_sample, d2_dbscan_labels)
d2_metrics = pd.DataFrame([
    {"dataset": "D2", "algorithm": "K-means", "clusters": best_k_d2, "silhouette": s_km, "davies_bouldin": db_km, "evaluated_points": len(d2_kmeans_labels), "noise_points": 0},
    {"dataset": "D2", "algorithm": "DBSCAN", "clusters": len(set(d2_dbscan_labels)) - (1 if -1 in d2_dbscan_labels else 0), "silhouette": s_db, "davies_bouldin": db_db, "evaluated_points": kept_db, "noise_points": noise_db},
])
d2_metrics.to_csv(OUT / "d2_clustering_metrics.csv", index=False)
d2_metrics

In [ ]:
d2_k_profile, d2_k_cat = profile_sample(d2.iloc[sample_idx2].reset_index(drop=True), d2_kmeans_labels, "d2_kmeans")
d2_db_profile, d2_db_cat = profile_sample(d2.iloc[sample_idx2].reset_index(drop=True), d2_dbscan_labels, "d2_dbscan")
print("D2 K-means profile")
display(d2_k_profile)
print("D2 DBSCAN profile")
display(d2_db_profile)

## Tổng hợp chỉ số và báo cáo

In [ ]:
all_metrics = pd.concat([d1_metrics, d2_metrics], ignore_index=True)
all_metrics.to_csv(OUT / "clustering_metrics.csv", index=False)
pd.DataFrame({"sample_index": sample_idx, "kmeans_cluster": d1_kmeans_labels, "dbscan_cluster": d1_dbscan_labels}).to_csv(OUT / "d1_cluster_assignments_sample.csv", index=False)
pd.DataFrame({"user_id": d2.iloc[sample_idx2]["user_id"].to_numpy(), "kmeans_cluster": d2_kmeans_labels, "dbscan_cluster": d2_dbscan_labels}).to_csv(OUT / "d2_cluster_assignments_sample.csv", index=False)

with PdfPages(OUT / "bao-cao-bai4.pdf") as pdf:
    for title, image_name in [("D1 - Chọn k", "d1_k_selection.png"), ("D2 - Chọn k", "d2_k_selection.png")]:
        fig = plt.figure(figsize=(11.7, 8.3))
        fig.text(0.07, 0.93, title, fontsize=18, weight="bold")
        image = plt.imread(FIG / image_name)
        plt.imshow(image)
        plt.axis("off")
        pdf.savefig(fig, bbox_inches="tight")
        plt.close(fig)
    fig, ax = plt.subplots(figsize=(11.7, 8.3))
    ax.axis("off")
    ax.set_title("Bài 4 - Đánh giá gom cụm", loc="left", fontsize=18, weight="bold", pad=20)
    table = ax.table(cellText=all_metrics.round(4).astype(str).values, colLabels=all_metrics.columns, loc="center", cellLoc="center")
    table.auto_set_font_size(False)
    table.set_fontsize(9)
    table.scale(1, 2)
    ax.text(0, 0.08, "Silhouette càng lớn càng tốt; Davies-Bouldin càng nhỏ càng tốt. DBSCAN loại điểm nhiễu (-1) khỏi việc tính hai chỉ số.", fontsize=10, wrap=True)
    pdf.savefig(fig, bbox_inches="tight")
    plt.close(fig)
    for dataset, profile, cat_profile in [("D1 K-means", d1_k_profile, d1_k_cat), ("D2 K-means", d2_k_profile, d2_k_cat)]:
        fig, axes = plt.subplots(2, 1, figsize=(11.7, 8.3), gridspec_kw={"height_ratios": [1, 1.4]})
        axes[0].axis("off")
        axes[0].set_title(f"{dataset} - Profiling số", loc="left", fontsize=15, weight="bold")
        axes[0].table(cellText=profile.round(2).astype(str).values, colLabels=profile.columns, rowLabels=profile.index, loc="center", cellLoc="center")
        axes[1].axis("off")
        axes[1].set_title(f"{dataset} - Giá trị hạng mục phổ biến", loc="left", fontsize=13)
        axes[1].table(cellText=cat_profile.astype(str).values, colLabels=cat_profile.columns, rowLabels=cat_profile.index, loc="center", cellLoc="center")
        pdf.savefig(fig, bbox_inches="tight")
        plt.close(fig)
print("Đã xuất:", OUT / "bao-cao-bai4.pdf")
display(all_metrics)